# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Emmamems18/flyrank-ml-internship-my-submission-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [4]:
df.sample(5)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
13871,content_914ac4179d4e,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,3829.0,24004.0,...,15000-25000,0.14,24.4,5.0,8.57,0.0,good,page_3_5,stable,-6.8
3753,content_cb93633967cf,client_19581e27de,50.0,0.01,LOW,0.0,keyword article,informational,NaN,NaN,...,NaN,0.00,19.2,0.0,0.00,0.0,moderate,striking,stable,-1.0
20086,content_45d9fafd55d9,client_98a3ab7c34,0.0,0.00,LOW,0.0,keyword article,informational,2765.0,16512.0,...,15000-25000,0.00,32.1,0.0,100.00,0.0,low,page_3_5,down,-60.0
22395,content_20f30ca9c2b1,client_f369cb89fc,10.0,0.00,LOW,0.0,keyword article,transactional,2615.0,15694.0,...,15000-25000,0.00,4.8,50.0,50.00,0.0,low,page_1,down,-50.0
1780,content_1a85f0009a31,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,3590.0,22370.0,...,15000-25000,0.22,21.9,10.0,6.25,0.0,moderate,page_3_5,up,160.6


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

ML Task Type: Scoring / Ranking (or Binary Classification as a baseline)

Why this task type:
SEO teams have limited time and cannot review thousands of pages at once. A simple classification model (e.g., predicting "will reach Page 1: Yes/No") tells us if a page can move, but Scoring / Ranking orders all 10,201 striking-distance candidates from highest expected traffic impact to lowest. This gives the team a prioritized queue so they can focus resources on the top 5% or 10% highest-yield opportunities first.

In [5]:
# This cell is for CODE (numbers, a query, a check).

# Filter for striking distance pages (positions 11-30)
striking_df = df[(df["avg_position"] >= 11) & (df["avg_position"] <= 30)].copy()

# Create a proxy Opportunity Score: higher search volume + closer to Page 1 = higher rank potential
# Scale position so position 11 gets higher weight than position 30
striking_df["opportunity_score"] = striking_df["search_volume"] * (
    31 - striking_df["avg_position"]
)

# Sort candidates by opportunity score to view top-ranked recommendations
ranked_candidates = striking_df[
    [
        "content_id",
        "avg_position",
        "search_volume",
        "ctr",
        "opportunity_score",
    ]
].sort_values(by="opportunity_score", ascending=False)

print("Top 5 Highest-Ranked Striking Distance Candidates:")
print(ranked_candidates.head())
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

Top 5 Highest-Ranked Striking Distance Candidates:
                 content_id  avg_position  search_volume   ctr  \
16585  content_19bdaa296a9b          13.5        33100.0  0.00   
2553   content_eb1510f4b5f1          14.7        33100.0  0.00   
5287   content_8ca50876b0df          18.3        40500.0  0.03   
10466  content_71f8734aebe2          15.0        27100.0  0.00   
9217   content_12e48d4b449d          17.0        27100.0  0.00   

       opportunity_score  
16585           579250.0  
2553            539530.0  
5287            514350.0  
10466           433600.0  
9217            379400.0  


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: Opportunity Score (or Priority Class)
Source: A defined rule (Proxy)

Explanation: Because this dataset is a single snapshot in time, we do not have longitudinal data showing which pages successfully moved to Page 1 after an update. Therefore, we cannot use an observed outcome. Instead, the target is a defined rule (a proxy): an Opportunity Score that multiplies search volume by proximity to Page 1. For classification purposes, this continuous score can be converted into a binary label, flagging the top 20% of scores as "High Priority" (1) and the rest as "Standard" (0).

In [6]:
# This cell is for CODE (numbers, a query, a check).

# Re-create the striking distance subset and the opportunity score
striking_df = df[(df["avg_position"] >= 11) & (df["avg_position"] <= 30)].copy()
striking_df["opportunity_score"] = striking_df["search_volume"] * (
    31 - striking_df["avg_position"]
)

# Create the Target label (Defined Rule): Top 20% become "High Priority" (Class 1)
threshold = striking_df["opportunity_score"].quantile(0.80)
striking_df["target_priority_class"] = np.where(
    striking_df["opportunity_score"] >= threshold, 1, 0
)

# Check the distribution of our new target label
class_counts = striking_df["target_priority_class"].value_counts()

print(f"Target Label Source: Defined Rule (Top 20% of Opportunity Score)")
print(f"Score Threshold to be 'High Priority': {threshold:,.0f}")
print("-" * 40)
print(f"High Priority Candidates (Class 1): {class_counts[1]:,} pages")
print(f"Standard Candidates (Class 0): {class_counts[0]:,} pages")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Target Label Source: Defined Rule (Top 20% of Opportunity Score)
Score Threshold to be 'High Priority': 398
----------------------------------------
High Priority Candidates (Class 1): 1,987 pages
Standard Candidates (Class 0): 8,214 pages


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric: Precision (for the "High Priority" Class 1).

Why this metric: Our target label is imbalanced (only ~20% of pages are High Priority). If we used Accuracy, a model could just guess "Standard" every time and be 80% accurate but entirely useless. Because SEO resources are expensive, our biggest business risk is a False Positive (paying an editor to rewrite a page that won't drive traffic). Precision directly measures this: out of all the pages the model claims are High Priority, how many actually are?

What number means 'good': Because the base rate of High Priority pages is about 19.5%, a random guess would give us 19.5% precision. A model achieving a precision of >70% on Class 1 would be considered "good," ensuring that 7 out of 10 editorial recommendations are genuinely high-yield targets.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Calculate the baseline precision (if we just guessed randomly)
total_candidates = len(striking_df)
high_priority_count = striking_df["target_priority_class"].sum()

# Baseline precision is just the percentage of the positive class in the dataset
baseline_precision = (high_priority_count / total_candidates) * 100

print("Class Imbalance & Model Baseline:")
print(f"Total Striking Distance Candidates: {total_candidates:,}")
print(f"High Priority Candidates (Class 1): {high_priority_count:,}")
print("-" * 45)
print(f"Baseline Precision (Random Guess): {baseline_precision:.1f}%")
print(f"Target Precision for a 'Good' Model: > 70.0%")
print("\nConclusion: The model must beat the 19.5% baseline significantly to be useful.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Class Imbalance & Model Baseline:
Total Striking Distance Candidates: 10,201
High Priority Candidates (Class 1): 1,987
---------------------------------------------
Baseline Precision (Random Guess): 19.5%
Target Precision for a 'Good' Model: > 70.0%

Conclusion: The model must beat the 19.5% baseline significantly to be useful.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One unique content page (content_id) currently sitting in striking distance (ranking positions 11–30).

Dataframe Slice Summary:

Filtered Subset: 10,201 rows representing candidates eligible for optimization.

Granularity: Each row contains page-level metrics (avg_position, search_volume, ctr, word_count, days_since_last_update, engagement_rate) alongside our defined proxy target (target_priority_class).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Select key features and the target label for the striking distance slice
slice_cols = [
    "content_id",
    "avg_position",
    "search_volume",
    "ctr",
    "word_count",
    "days_since_last_update",
    "engagement_rate",
    "target_priority_class",
]

# Extract slice and check shape
striking_slice = striking_df[slice_cols].copy()

print(f"Unit of Analysis: 1 row = 1 content page (content_id)")
print(f"Slice Shape: {striking_slice.shape[0]:,} rows × {striking_slice.shape[1]} columns\n")

# Display first 5 rows of the actual dataframe slice
striking_slice.head(10)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

Unit of Analysis: 1 row = 1 content page (content_id)
Slice Shape: 10,201 rows × 8 columns



,content_id,avg_position,search_volume,ctr,word_count,days_since_last_update,engagement_rate,target_priority_class
1,content_a1fb4e703a9e,20.3,90.0,0.05,2481.0,25,0.00,1
7,content_a63219c6e95a,21.2,590.0,0.06,NaN,22,3.57,1
14,content_91067a14431a,27.1,0.0,0.00,2802.0,104,0.00,0
18,content_0b360eb9db55,11.4,30.0,0.14,2945.0,20,2.33,1
21,content_9d548144b06d,12.6,0.0,0.00,3455.0,20,0.00,0
23,content_2da6ae9d0882,13.9,0.0,0.34,NaN,20,0.00,0
26,content_72c5c2d73e5a,30.0,0.0,0.12,2686.0,13,0.00,0
29,content_ba8e51f13800,16.5,10.0,0.00,1626.0,20,7.14,0
32,content_5eeba5d398f2,28.3,0.0,0.66,2949.0,20,0.00,0
35,content_1a28b25c7128,24.8,0.0,0.03,6877.0,104,0.00,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why Machine Learning Beats a Fixed Rule (if/else Statements):

**Complex Non-Linear Interactions**: No single metric guarantees success. A page with low search volume at position 11 might be a quicker win than a page with high search volume sitting at position 29. Simple if statements cannot dynamically balance trade-offs across 5+ metrics simultaneously.

**Real-World Missing Data (NaNs)**: As seen in our dataset, features like word_count contain missing values (NaN). Hardcoded if/else rules break or skip rows when encountering missing data, whereas ML models (like XGBoost or Random Forests) handle missing values natively.

**Arbitrary Thresholds vs. Learned Weights**: Manual rules rely on guessed cutoffs (e.g., if word_count > 2000 and days_since_last_update > 180). ML automatically learns optimal split points and relative feature importance directly from patterns in the data, preventing human bias from discarding viable candidates.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# 1. Demonstrate missing data messiness that breaks simple if-statements
missing_summary = striking_slice.isna().sum()

print("Missing Values Count Across Striking Distance Features:")
print(missing_summary[missing_summary > 0])
print("-" * 50)

# 2. Show feature interaction complexity (High volume pages aren't always updated recently)
complex_subset = striking_slice[
    (striking_slice["search_volume"] > 1000)
    & (striking_slice["avg_position"] <= 15)
]

print(f"Pages with Search Vol > 1,000 AND Position <= 15: {len(complex_subset):,}")
print(
    "Missing word counts in this high-potential subset:"
    f" {complex_subset['word_count'].isna().sum()}"
)
print(
    "Average days since update for this subset:"
    f" {complex_subset['days_since_last_update'].mean():.1f} days"
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

Missing Values Count Across Striking Distance Features:
search_volume     290
word_count       2217
dtype: int64
--------------------------------------------------
Pages with Search Vol > 1,000 AND Position <= 15: 57
Missing word counts in this high-potential subset: 22
Average days since update for this subset: 37.1 days


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.